# Galassi 2012 - Gate A: Protocol and Data Ingestion

This notebook implements Gate A of the validation pipeline. It handles protocol definition, data ingestion (with placeholder detection), visualization, and checkpointing.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import hashlib

# Try to mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/H2_Storage'
except ImportError:
    # Fallback for local execution
    BASE_DIR = 'H2_Storage'
    print("Not running in Colab. Using local directory: " + BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, 'validation/data/galassi_2012')
NOTEBOOK_DIR = os.path.join(BASE_DIR, 'validation/notebooks')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
FIGURES_DIR = os.path.join(BASE_DIR, 'figures')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')

# Ensure directories exist
for d in [DATA_DIR, NOTEBOOK_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

## A) Write Protocol JSON

In [ ]:
protocol_data = {
    "H2_101": {"Pin_bar": 0.26, "Pfin_bar": 717, "t_fill_s": 330, "Tamb_C": 16, "Tini_C": 21},
    "H2_250": {"Pin_bar": 0.26, "Pfin_bar": 718, "t_fill_s": 245, "Tamb_C": 18, "Tini_C": 21}
}

protocol_path = os.path.join(DATA_DIR, 'protocol_table1.json')
with open(protocol_path, 'w') as f:
    json.dump(protocol_data, f, indent=4)
print(f"Created {protocol_path}")

## B & C) Data Ingestion and Placeholder Detection

In [ ]:
expected_files = ['fig5_TC5_H2_101_exp.csv', 'fig5_TC5_H2_250_exp.csv']
loaded_data = {}
placeholder_flags = {}
file_hashes = {}

def calculate_sha256(filepath):
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

for filename in expected_files:
    filepath = os.path.join(DATA_DIR, filename)
    
    # Check if missing
    if not os.path.exists(filepath):
        print(f"Missing {filename}, creating placeholder.")
        # Create placeholder
        # 10 points, linear temperature curve
        # time_s, T_K, is_placeholder
        df_placeholder = pd.DataFrame({
            'time_s': np.linspace(0, 300, 10),
            'T_K': np.linspace(294, 350, 10), # Random linear rise
            'is_placeholder': 1
        })
        df_placeholder.to_csv(filepath, index=False)
        
        # Write warning
        with open(os.path.join(DATA_DIR, 'PLACEHOLDER_WARNING.txt'), 'w') as f:
            f.write("WARNING: Placeholder data generated for missing digitized files.\n")
            
    # Load and detect placeholder status
    df = pd.read_csv(filepath)
    
    is_placeholder = False
    if 'is_placeholder' in df.columns:
        if (df['is_placeholder'] == 1).any():
            is_placeholder = True
    
    if len(df) <= 10:
        is_placeholder = True
        
    loaded_data[filename] = df
    placeholder_flags[filename] = is_placeholder
    file_hashes[filename] = calculate_sha256(filepath)

## D) Plotting

In [ ]:
plt.figure(figsize=(10, 6))
for filename, df in loaded_data.items():
    label = filename.replace('.csv', '')
    if placeholder_flags[filename]:
        label += " (PLACEHOLDER)"
    plt.plot(df['time_s'], df['T_K'], label=label)

plt.xlabel('Time (s)')
plt.ylabel('Temperature (K)')
plt.title('Galassi 2012 Fig 5 TC5 Extracted Data')
plt.legend()
plt.grid(True)
figure_path = os.path.join(FIGURES_DIR, 'galassi2012_fig5_TC5_extracted.png')
plt.savefig(figure_path)
print(f"Saved plot to {figure_path}")
plt.show()

## E) Save Checkpoint

In [ ]:
checkpoint_data = {
    'protocol': protocol_data,
    'dataframes': loaded_data,
    'placeholder_flags': placeholder_flags,
    'hashes': file_hashes
}

checkpoint_path = os.path.join(CHECKPOINTS_DIR, 'galassi2012_gateA.pkl')
with open(checkpoint_path, 'wb') as f:
    pickle.dump(checkpoint_data, f)
print(f"Saved checkpoint to {checkpoint_path}")

## F) Final Summary

In [ ]:
print("\n--- SUMMARY ---")
print("Protocol Values:")
print(json.dumps(protocol_data, indent=2))
print("\nFile Status:")
any_placeholder = False
for filename, is_ph in placeholder_flags.items():
    status = "PLACEHOLDER" if is_ph else "REAL"
    print(f"{filename}: {status}")
    if is_ph:
        any_placeholder = True

if any_placeholder:
    print("\nSTOP: Provide digitized Fig.5 CSVs before running Gate B.")